In [ ]:
import pandas as pd
import numpy as np
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch
from sklearn.metrics import roc_auc_score
import random
import glob

In [ ]:
all_files = glob.glob(os.path.join('../boilerplates_for_spaces_for_patient_checks', "*.csv"))

if not all_files:
    print(f"No CSV files found in the directory: {directory_path}")

list_of_dfs = []
for file_path in all_files:
    try:
        df = pd.read_csv(file_path)
        list_of_dfs.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

if list_of_dfs:
    combined_df = pd.concat(list_of_dfs, ignore_index=True)
else:
    print("No DataFrames could be successfully loaded.")


In [ ]:
validation_set = combined_df
#validation_set = validation_set[validation_set.split.str.contains('test')] # no, run for all
validation_set.split.value_counts()

In [ ]:

validation_set.info()


In [ ]:
validation_set.this_space.nunique()

In [ ]:
validation_set.patient_summary.nunique()

In [ ]:
validation_set = validation_set[~validation_set.trial_boilerplate_text.isnull()]

In [ ]:
validation_set.info()

In [ ]:
validation_set['pt_trial_pair'] = validation_set['this_space'] + "\nNow here is the patient summary:" + validation_set['patient_summary']


In [ ]:
pruned_set = validation_set

In [ ]:
pruned_set['boilerplate_pair'] = "Patient history: " + pruned_set['patient_boilerplate_text'] + "\nTrial exclusions:" + pruned_set['trial_boilerplate_text']
pruned_set = pruned_set[~pruned_set.boilerplate_pair.isnull()]

In [ ]:
pruned_set = pruned_set[['boilerplate_pair', 'exclusion_result']].rename(columns={'boilerplate_pair':'text','exclusion_result':'label'})

In [ ]:
from transformers import pipeline, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('../../../v20_models/boilerplatechecker')
pipe = pipeline('text-classification', '../../../v20_models/boilerplatechecker', tokenizer=tokenizer, truncation=True, padding='max_length', max_length=3196, device='cuda') 
predictions = pipe(pruned_set.text.tolist())

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
predictions_frame = pd.DataFrame(predictions)
predictions_frame['score'] = np.where(predictions_frame.label=='NEGATIVE', 1 - predictions_frame.score, predictions_frame.score)
predictions_frame['logit_score'] = np.log(predictions_frame.score + 1e-6 / (1 - predictions_frame.score + 1e-6))


In [ ]:
pruned_set = pd.concat([pruned_set.rename(columns={'label':'exclusion_result'}), predictions_frame], axis=1)
pruned_set.info()

In [ ]:
pruned_set.to_csv('boilerplate_patient_centric_results.csv')

In [ ]:
import pandas as pd
pruned_set = pd.read_csv("boilerplate_patient_centric_results.csv")

In [ ]:
from sklearn.metrics import roc_auc_score
# checker results on round 2 generated data ('round 3 data')
roc_auc_score(pruned_set.exclusion_result, pruned_set.score)

In [ ]:
from utils_102023 import eval_model
print(pruned_set.shape[0])
eval_model(pruned_set.score, pruned_set.exclusion_result, graph=True)

In [ ]:
import seaborn as sns
sns.distplot(pruned_set.score)

In [ ]:
import numpy as np
np.min(pruned_set.score)

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(np.where(pruned_set.exclusion_result == 0.0, 'NEGATIVE','POSITIVE'), pruned_set.label)

In [ ]:
pruned_set.info()